In [34]:
from dotenv import load_dotenv
load_dotenv()

True

In [35]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.prompts import PromptTemplate

### doc loading

In [36]:
loader=PyPDFLoader("../data/data_science_syllabus.pdf")
doc=loader.load()
len(doc)

10

### text splitting

In [37]:
from gitdb.fun import chunk_size
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
pdf_chunks=splitter.split_documents(doc)

### embedding

In [38]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector = embeddings.embed_documents("document")


### vector DB

In [39]:

vector_store=Chroma.from_documents(
    documents=pdf_chunks,
    embedding=embeddings,
    
)

### llm

In [40]:
llm=ChatGroq(model="openai/gpt-oss-20b")

### context generator for the llm

In [41]:

def get_context(query:str):
    data=vector_store.similarity_search(query)
    context=""
    for d in data:
        context=context+d.page_content+"/n"
    
    return{
        "context":context,
        "question":query
    }

### prompt template

In [42]:
prompt=PromptTemplate.from_template('''You are a supportive assistant and you do help the user by giving answer from the context based on user's question if you dont know the answer just say "No Context Provided"

context:{context}
question:{question}


''')

### rag chain

In [43]:
rag_chain=get_context|prompt|llm

In [47]:
res=rag_chain.invoke("what is virat kohli's total number of centuries ?")

In [48]:
print(res.content)

No Context Provided
